# lotte_stance 3-class 분류기 학습 (Google Colab)

lotte-insight 프로젝트 — `training/train_stance_classifier.py` Colab 실행용 노트북

**모델:** `monologg/koelectra-small-v3-discriminator` fine-tuning
**분류 방식:** 3-class CrossEntropyLoss (negative / neutral / positive)
**학습 데이터:** is_lotte_related=True 행만 (~6,739건 예상)
**목표:** val macro F1 ≥ 0.70
**예상 소요:** T4 GPU 기준 약 5~10분

**사전 준비**
- 런타임 유형: T4 GPU (런타임 → 런타임 유형 변경)
- 업로드할 파일: `training/data/labeled_titles.csv`, `training/data/labeled_players.csv`

In [ ]:
# 1. GPU 확인
import torch
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('CUDA:', torch.version.cuda)
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    print('[WARN] GPU 없음 — CPU 학습 시 약 2~3시간 소요')

In [ ]:
# 2. 레포 클론 및 의존성 설치
GITHUB_REPO_URL = 'https://github.com/JoeYunHa/Lotte_Insight.git'

!git clone {GITHUB_REPO_URL} /content/lotte-insight
%cd /content/lotte-insight/training
!pip install -q transformers torch scikit-learn pandas numpy

In [ ]:
# 3. 학습 데이터 업로드
import os
from google.colab import files

DATA_DIR = '/content/lotte-insight/training/data'
os.makedirs(DATA_DIR, exist_ok=True)

uploaded = files.upload()

for fname, content in uploaded.items():
    dst = f'{DATA_DIR}/{os.path.basename(fname)}'
    with open(dst, 'wb') as f:
        f.write(content)
    print(f'저장 완료: {dst}  ({len(content):,} bytes)')

In [ ]:
# 4. 학습 데이터 분포 확인
import pandas as pd

STANCE_LABELS = ['negative', 'neutral', 'positive']
totals = {l: 0 for l in STANCE_LABELS}

for fname in ['labeled_titles.csv', 'labeled_players.csv']:
    path = f'{DATA_DIR}/{fname}'
    if not os.path.exists(path):
        print(f'[MISSING] {fname}')
        continue
    df = pd.read_csv(path, encoding='utf-8-sig')
    df_related = df[df['is_lotte_related'].astype(str).str.lower().isin({'true', '1', 'yes'})].copy()
    df_labeled = df_related.dropna(subset=['lotte_stance'])
    df_labeled = df_labeled[df_labeled['lotte_stance'].isin(STANCE_LABELS)]
    print(f'{fname}: 총 {len(df)}행 → 학습 가능 {len(df_labeled)}행 (제외 {len(df) - len(df_labeled)})')
    for label in STANCE_LABELS:
        cnt = (df_labeled['lotte_stance'] == label).sum()
        totals[label] += cnt
        print(f'  {label}: {cnt}')
    print()

total = sum(totals.values())
print('=== 합산 (학습 예정) ===')
for label in STANCE_LABELS:
    print(f'  {label}: {totals[label]} ({totals[label]/total:.1%})')
print(f'  합계: {total}')

In [ ]:
# 5. 학습 실행 (T4 GPU 기준 약 5~10분)
!python train_stance_classifier.py \
    --data-dir /content/lotte-insight/training/data \
    --output-dir /content/lotte-insight/training/models/stance_koelectra \
    --epochs 5 \
    --lr 5e-5 \
    --batch 16

In [ ]:
# 6. 학습 결과 확인
import json, os

MODEL_DIR = '/content/lotte-insight/training/models/stance_koelectra'
config_path = f'{MODEL_DIR}/stance_config.json'

if os.path.exists(config_path):
    with open(config_path) as f:
        cfg = json.load(f)
    print('저장된 라벨 매핑:', cfg['label2id'])
    print('배포 시 STANCE_MODEL_DIR=/app/models/stance_koelectra 로 맞추면 backend가 동일 매핑으로 로드합니다.')
else:
    print('[ERROR] stance_config.json 없음 — 학습 실패 여부 확인')

print('\n모델 파일 목록:')
for fn in sorted(os.listdir(MODEL_DIR)):
    size = os.path.getsize(f'{MODEL_DIR}/{fn}')
    print(f'  {fn:<40} {size:>10,} bytes')

In [ ]:
# 7. Smoke test
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_DIR = '/content/lotte-insight/training/models/stance_koelectra'
STANCE_LABELS = ['negative', 'neutral', 'positive']

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR)
model.eval()

def predict_stance(title, snippet=''):
    enc = tokenizer(
        title, snippet[:300].strip(),
        truncation='only_second', padding='max_length',
        max_length=128, return_tensors='pt',
    )
    with torch.no_grad():
        logits = model(**enc).logits[0]
        probs = torch.softmax(logits, dim=-1).tolist()
    best_idx = max(range(len(probs)), key=lambda i: probs[i])
    return {'label': STANCE_LABELS[best_idx], 'confidence': round(probs[best_idx], 4)}

test_cases = [
    ('나균안, 6이닝 2실점 호투로 시즌 5승…롯데 3연패 탈출', '나균안이 두산전 6이닝 2실점 호투로 시즌 5승을 달성했다.', 'positive'),
    ('롯데 불펜 붕괴…9회 4점 내주며 역전패', '롯데 불펜진이 9회에 무너지며 경기를 뒤집혔다.', 'negative'),
    ('롯데 자이언츠, 내일 두산 원정 예고', '롯데는 내일 두산 베어스와 잠실 원정 경기를 치를 예정이다.', 'neutral'),
    ('전준우 햄스트링 부상, 2주 결장 전망', '전준우가 햄스트링 부상으로 2주간 결장할 전망이다.', 'neutral'),
    ('롯데, 외국인 투수 방출…새 용병 물색 중', '롯데가 부진 외국인 투수를 방출하고 대체 용병을 찾고 있다.', 'negative'),
]

print('=== Smoke Test ===')
passed = 0
for title, snippet, expected in test_cases:
    result = predict_stance(title, snippet)
    ok = result['label'] == expected
    mark = 'O' if ok else 'X'
    if ok:
        passed += 1
    print(f'{mark} [{result["label"]:>8}] conf={result["confidence"]:.3f}  (기대: {expected})')
    print(f'   {title}')
    print()

print(f'결과: {passed}/{len(test_cases)} 통과')

In [ ]:
# 8. 모델 다운로드 (Colab → 로컬)
import shutil, zipfile
from google.colab import files

MODEL_DIR = '/content/lotte-insight/training/models/stance_koelectra'
ZIP_PATH = '/content/stance_koelectra.zip'

shutil.make_archive('/content/stance_koelectra', 'zip', MODEL_DIR)
print(f'압축 완료: {ZIP_PATH}')
with zipfile.ZipFile(ZIP_PATH) as z:
    for name in sorted(z.namelist()):
        info = z.getinfo(name)
        print(f'  {name:<45} {info.file_size:>10,} bytes')
files.download(ZIP_PATH)

## 다운로드 후 로컬 배치

```
stance_koelectra.zip 압축 해제
  → training/models/stance_koelectra/
```

필수 파일:
- `config.json`, `pytorch_model.bin` (또는 `model.safetensors`)
- `tokenizer_config.json`, `vocab.txt`
- `stance_config.json` ← 라벨 매핑

배치 완료 후 `STANCE_MODEL_DIR=/app/models/stance_koelectra` 환경변수를 설정하면
`backend/models/stance_classifier.py`의 `_runtime`이 동일 모델을 로드합니다.
`news_collector.py`의 `classify_stance()` 호출 시 이 모델이 사용됩니다.